# 03. Indeks terbalik, TF-IDF, dan BM25 (Fase 3)

**Tujuan:** membangun pencarian berbasis kata yang cepat dan bisa dijelaskan, lalu mengukurnya dengan `qrels.csv`.

**Yang dipelajari:** inverted index, Boolean search, TF-IDF dengan cosine, rumus BM25 dan arti parameter `k1` dan `b`,
serta metrik Precision@K, Recall@K, MRR, dan NDCG.

In [1]:
import sys, warnings
sys.path.insert(0, "..")          # agar folder nlp_engine/ bisa diimpor
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from nlp_engine import *
from nlp_engine.preprocessing import normalize_text, tokenize, stem, STOPWORDS
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 160)
jobs, queries, qrels, ex = load_data("..")
job_ids = jobs.job_id.tolist()
print(f"{len(jobs)} lowongan, {len(queries)} kueri uji, {len(ex.names)} skill di kamus")

368 lowongan, 30 kueri uji, 56 skill di kamus


## 1. Dari teks ke token
Setiap lowongan dipraproses dengan jalur yang sama seperti di notebook 02 (termasuk kanonisasi skill).

In [2]:
texts = job_text(jobs)
docs_tok = [preprocess(t, ex) for t in texts]
print(texts[1][:160], "...")
print(docs_tok[1][:25])

Data Engineer . Data Engineer . Python SQL ETL . Bank Mitra Rakyat membuka kesempatan untuk posisi Data Engineer di Tangerang. Tugas utama: memastikan kualitas  ...
['data', 'engineer', 'data', 'engineer', 'sk_python', 'sk_sql', 'sk_etl', 'bank', 'mitra', 'rakyat', 'buka', 'kesempat', 'posisi', 'data', 'engineer', 'tangerang', 'tugas', 'utama', 'asti', 'kualitas', 'ketersedia', 'data', 'tim', 'analis', 'merancang']


## 2. Inverted index
Struktur data yang menyimpan hubungan **kata → dokumen**. Dengan ini, mencari lowongan yang memuat kata tertentu
tidak perlu membaca semua dokumen.

In [3]:
ix = InvertedIndex(docs_tok)
print(f"{ix.N} dokumen, {len(ix.postings)} term unik, rata-rata {ix.avgdl:.1f} token per dokumen")
post = ix.postings["sk_python"]
print("sk_python muncul di", len(post), "lowongan. Lima pertama (doc_id: frekuensi):", dict(list(post.items())[:5]))

368 dokumen, 796 term unik, rata-rata 46.2 token per dokumen
sk_python muncul di 58 lowongan. Lima pertama (doc_id: frekuensi): {1: 2, 17: 1, 24: 2, 29: 2, 39: 2}


### Boolean search
AND mempersempit, OR memperluas. Sederhana, tetapi tidak mengurutkan hasil.

In [4]:
and_hits = ix.boolean_and(["sk_python", "sk_sql"])
or_hits  = ix.boolean_or(["sk_python", "sk_sql"])
print("Python AND SQL:", len(and_hits), "lowongan   |   Python OR SQL:", len(or_hits), "lowongan")
jobs.loc[and_hits[:5], ["job_id", "job_title", "required_skills"]]

Python AND SQL: 18 lowongan   |   Python OR SQL: 103 lowongan


,job_id,job_title,required_skills
1,J0002,Data Engineer,Python;SQL;ETL
17,J0018,Data Engineer,Python;SQL;ETL
39,J0040,Data Engineer,Python;SQL;ETL;Linux
45,J0046,Junior Data Engineer,Python;SQL;ETL
65,J0066,Business Intelligence Analyst,SQL;Power BI;Microsoft Excel;Analisis Data


## 3. TF-IDF dan cosine similarity
Bobot kata = seberapa sering muncul di dokumen (TF) dikali seberapa langka di seluruh korpus (IDF). Kata yang ada di
hampir semua lowongan (misalnya "tim") tidak membantu membedakan lowongan.

In [5]:
tfidf = TfidfRetriever(ix)
idf_sorted = sorted(tfidf.idf.items(), key=lambda kv: kv[1])
print("IDF terendah (paling umum):", [(t, round(v, 2)) for t, v in idf_sorted[:6]])
print("IDF tertinggi (paling langka):", [(t, round(v, 2)) for t, v in idf_sorted[-6:]])

IDF terendah (paling umum): [('utama', 1.34), ('tugas', 1.35), ('didi', 1.46), ('minimal', 1.46), ('s1', 1.46), ('tim', 1.59)]
IDF tertinggi (paling langka): [('report', 6.91), ('weekly', 6.91), ('formance', 6.91), ('cross', 6.91), ('apps', 6.91), ('disukai', 6.91)]


## 4. BM25
TF-IDF membiarkan kata yang diulang-ulang menang terus dan tidak mempertimbangkan panjang dokumen. BM25 memperbaikinya:

$$\text{skor}(q,d)=\sum_{t\in q}\text{idf}(t)\cdot\frac{tf\,(k_1+1)}{tf+k_1\,(1-b+b\,|d|/\text{avgdl})}$$

- `k1` mengatur seberapa cepat tambahan frekuensi kata berhenti menambah skor (jenuh).
- `b` mengatur seberapa kuat panjang dokumen menurunkan skor (0 = abaikan panjang, 1 = penuh).

In [6]:
bm25 = BM25(ix, k1=1.5, b=0.75)
q = "Saya menguasai Python, SQL, Pandas, dan visualisasi data. Mencari pekerjaan fresh graduate di bidang analisis data."
qt = preprocess(q, ex); print("token kueri:", qt)
top = bm25.search(qt, 8)
out = jobs.loc[top, ["job_id", "job_title", "experience_level"]].copy(); out["skor BM25"] = bm25.scores(qt)[top].round(2); out

token kueri: ['uasai', 'sk_python', 'sk_sql', 'sk_pandas', 'sk_visualisasi_data', 'fresh', 'graduate', 'sk_analisis_data']


,job_id,job_title,experience_level,skor BM25
65,J0066,Business Intelligence Analyst,Mid-level,11.97
82,J0083,Business Intelligence Analyst,Mid-level,11.65
13,J0014,Data Analyst (Fresh Graduate),Entry Level,11.62
156,J0157,Reporting Analyst (Fresh Graduate),Entry Level,11.43
330,J0331,Data Analyst (Fresh Graduate),Entry Level,11.24
121,J0122,Analis Riset Pasar,Entry Level,9.24
145,J0146,Backend Developer (Fresh Graduate),Entry Level,8.97
202,J0203,Business Intelligence Analyst Associate,Entry Level,8.94


### Cek silang dengan pustaka `rank_bm25`
Rumus IDF kedua implementasi berbeda sedikit, jadi yang kita bandingkan adalah kesamaan urutan teratas, bukan angka skor.

In [7]:
from rank_bm25 import BM25Okapi
ref = BM25Okapi(docs_tok, k1=1.5, b=0.75)
ref_top = list(np.argsort(-ref.get_scores(qt))[:10])
mine_top = bm25.search(qt, 10)
print("irisan top-10:", len(set(ref_top) & set(mine_top)), "dari 10")

irisan top-10: 9 dari 10


## 5. Mengukur dengan `qrels.csv`
30 kueri uji, tiap lowongan bernilai 2 (sangat relevan), 1 (cukup), atau tidak tercantum (0).
Metrik: **P@5**, **Recall@10**, **MRR@10**, **NDCG@5** (lihat penjelasannya di `nlp_engine/evaluation.py`).

In [8]:
exact = ExactKeyword(texts)
systems = {
    "E1 Kata persis":      lambda s: exact.search(s, 50),
    "E2 TF-IDF + cosine":  lambda s: tfidf.search(preprocess(s, ex), 50),
    "E3 BM25":             lambda s: bm25.search(preprocess(s, ex), 50),
}
rows = {}
for name, fn in systems.items():
    summary, _ = evaluate_system(fn, queries, qrels, job_ids)
    rows[name] = summary
pd.DataFrame(rows).T

,P@5,Recall@10,MRR@10,NDCG@5,NDCG@10
E1 Kata persis,0.873,0.747,1.000,0.858,0.764
E2 TF-IDF + cosine,0.893,0.847,0.944,0.839,0.803
E3 BM25,0.880,0.827,0.944,0.844,0.797


Yang perlu dibaca dengan hati-hati: dengan 30 kueri, selisih kecil antar metode belum tentu bermakna.
Lihat per kueri untuk tahu di mana sebuah metode kalah.

In [9]:
_, pq_e1 = evaluate_system(systems["E1 Kata persis"], queries, qrels, job_ids)
_, pq_e3 = evaluate_system(systems["E3 BM25"], queries, qrels, job_ids)
cmp = pd.DataFrame({"kueri": queries.set_index("query_id")["query"].str[:55], "NDCG@5 E1": pq_e1["NDCG@5"].round(2), "NDCG@5 BM25": pq_e3["NDCG@5"].round(2)})
cmp["selisih"] = (cmp["NDCG@5 BM25"] - cmp["NDCG@5 E1"]).round(2)
cmp.sort_values("selisih", ascending=False).head(8)

,kueri,NDCG@5 E1,NDCG@5 BM25,selisih
query_id,,,,
Q10,mengelola server dan deployment aplikasi di cloud,0.55,0.85,0.30
Q19,merekrut karyawan dan mengatur wawancara,0.72,1.00,0.28
Q09,membuat REST API dan mengelola database,0.85,1.00,0.15
Q26,pekerjaan remote untuk developer Python,0.43,0.55,0.12
Q13,mengelola kampanye iklan online dan media sosial,0.93,1.00,0.07
Q29,"saya punya skill Excel dan akuntansi, cari kerja di Sur",0.86,0.93,0.07
Q30,engineer senior berpengalaman membangun data pipeline,0.84,0.86,0.02
Q01,"Saya menguasai Python, SQL, Pandas, dan visualisasi dat",0.71,0.72,0.01


## 6. Pengaruh `k1` dan `b`

In [10]:
grid = []
for k1 in [0.8, 1.2, 1.5, 2.0]:
    for b in [0.0, 0.5, 0.75, 1.0]:
        m = BM25(ix, k1=k1, b=b)
        s, _ = evaluate_system(lambda x: m.search(preprocess(x, ex), 50), queries, qrels, job_ids)
        grid.append({"k1": k1, "b": b, "NDCG@5": s["NDCG@5"], "P@5": s["P@5"]})
g = pd.DataFrame(grid)
g.pivot(index="k1", columns="b", values="NDCG@5")

b,0.00,0.50,0.75,1.00
k1,,,,
0.8,0.829,0.837,0.841,0.838
1.2,0.831,0.832,0.845,0.846
1.5,0.831,0.839,0.844,0.855
2.0,0.843,0.846,0.852,0.857


> **Latihan 4.** Selisih antar sel pada tabel di atas kecil. Kenapa? (Petunjuk: lihat panjang dokumen di dataset, apakah variasinya besar?)
> Bandingkan dengan korpus yang dokumennya sangat beragam panjangnya.

> **Latihan 5.** Ubah `job_text()` di `nlp_engine/pipeline.py` sehingga judul hanya muncul sekali, lalu ukur ulang.
> Fitur apa saja yang perlu diberi bobot lebih? Ini cikal bakal *field weighting* (BM25F).

> **Latihan 6.** Tambahkan Boolean filter lokasi dan level **sebelum** peringkat, bandingkan dengan menjadikannya bagian dari skor.